# Phase 10 — Production Deployment and Operations

- **10A.** Repository and command inspection
- **10B.** Cloud decision and resource inventory
- **10C.** Production configuration
- **10D.** Object-storage artifact repository
- **10E.** Structured logging and reports
- **10F.** CI
- **10G.** Production container images
- **10H.** Registry publishing
- **10I.** Staging infrastructure and deployment
- **10J.** Scheduled inference/AQI pipeline
- **10K.** Incremental and retraining schedules
- **10L.** Monitoring, alerts, and stale-data checks
- **10M.** Production deployment
- **10N.** Rollback and failure testing
- **10O.** Documentation and final reports

## **10A.** Repository and operational-command inspection

Phase 10A validates that the existing application and MLOps workloads can be
operated non-interactively before cloud infrastructure or GitHub Actions are
created.

The inspection covers:

- serving applications
- Docker assets
- dependency management
- automated tests
- batch-pipeline commands
- Hopsworks commands
- artifact directories
- structured reports
- health endpoints
- existing workflow files

No deployment or cloud resource is created during this subphase.

In [1]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "pyproject.toml"
).exists(), "Could not resolve the project root."

print("Project root:", PROJECT_ROOT)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor


### Deployment asset inspection

The repository is inspected for the minimum assets required before selecting a
cloud deployment target:

- FastAPI Dockerfile
- Streamlit Dockerfile
- Docker Compose configuration
- locked Python dependencies
- environment template
- tests
- serving entry points
- reusable Phase 9 batch commands

In [2]:
from app.operations.repository_inspection import (
    inspect_expected_files,
)

import pandas as pd


file_inspections = inspect_expected_files()

file_inspection_df = pd.DataFrame(
    [
        {
            "name": item.name,
            "path": item.path,
            "exists": item.exists,
            "required": item.required,
        }
        for item in file_inspections
    ]
)

display(file_inspection_df)

,name,path,exists,required
0,FastAPI Dockerfile,Dockerfile,True,True
1,Streamlit Dockerfile,dashboard/Dockerfile,True,True
2,Docker Compose,compose.yaml,True,True
3,Python project configuration,pyproject.toml,True,True
4,Locked dependencies,uv.lock,True,True
5,Environment template,.env.example,True,True
6,FastAPI application,app/api/main.py,True,True
7,Streamlit application,dashboard/app.py,True,True
8,Live inference notebook,notebooks/06_live_inference_pipeline.ipynb,True,True
9,AQI pipeline notebook,notebooks/07_build_aqi_alert_pipeline.ipynb,False,True


### Operational command contract

Every production command must:

- run without interactive input
- produce a meaningful exit code
- use UTC timestamps
- write a structured report when it is a batch pipeline
- fail clearly
- avoid logging secrets
- remain safe to retry where applicable

Serving commands are long-running processes. Batch commands must terminate
after success or failure.

In [3]:
from app.operations.repository_inspection import (
    inspect_commands,
)


command_inspections = inspect_commands()

commands_df = pd.DataFrame(
    [
        {
            "name": item.name,
            "category": item.category,
            "available": item.available,
            "non_interactive": item.non_interactive,
            "command": item.command,
            "expected_report": item.expected_report,
        }
        for item in command_inspections
    ]
)

display(commands_df)

,name,category,available,non_interactive,command,expected_report
0,Run complete test suite,validation,True,True,uv run pytest -v,None
1,Run Ruff lint checks,validation,True,True,uv run ruff check .,None
2,Run Ruff formatting check,validation,True,True,uv run ruff format --check .,None
3,Start FastAPI,serving,True,True,uv run uvicorn app.api.main:app --host 0.0.0.0...,None
4,Start Streamlit,serving,True,True,uv run streamlit run dashboard/app.py --server...,None
5,Validate registry model resolution,mlops,True,True,uv run python -m app.pipelines.validate_regist...,reports/phase_9/production_model_resolution_re...
6,Run incremental feature synchronization,batch,True,True,uv run python -m app.pipelines.incremental_fea...,reports/phase_9/incremental_feature_report.json
7,Run historical backfill,batch,True,True,uv run python -m app.pipelines.historical_back...,reports/phase_9/historical_backfill_report.json
8,Build training dataset,mlops,True,True,uv run python -m app.pipelines.build_training_...,reports/phase_9/training_dataset_report.json
9,Run retraining eligibility,mlops,True,True,uv run python -m app.pipelines.retraining_cycle,reports/phase_9/automated_training_report.json


### Current artifact-storage boundary

The existing project stores validated artifacts under local directories such
as:

- `inference/runs`
- `inference/latest`
- `aqi/runs`
- `aqi/latest`
- `reports/phase_9`
- `models`

Local storage is suitable for development and Docker Compose, but independent
cloud services and scheduled jobs require durable shared storage.

Phase 10A records the current paths only. Object-storage implementation belongs
to a later Phase 10 subphase.

In [4]:
from app.operations.repository_inspection import (
    discover_artifact_directories,
)


artifact_summary = (
    discover_artifact_directories()
)

artifact_df = pd.DataFrame(
    [
        {
            "name": name,
            **details,
        }
        for name, details
        in artifact_summary.items()
    ]
)

display(artifact_df)

,name,path,exists,file_count
0,canonical_data,data/processed,True,3
1,training_data,data/training,True,8
2,models,models,True,30
3,inference_runs,inference/runs,True,48
4,inference_latest,inference/latest,False,0
5,aqi_runs,aqi/runs,True,30
6,aqi_latest,aqi/latest,True,6
7,phase_9_reports,reports/phase_9,True,10
8,phase_10_reports,reports/phase_10,True,3


### Existing health and serving endpoints

Deployment smoke tests will reuse the existing API and Streamlit health
contracts.

FastAPI must remain live even when forecast artifacts are missing or stale.
Readiness should represent artifact availability and freshness accurately.

In [5]:
from app.operations.repository_inspection import (
    HEALTH_ENDPOINTS,
)


health_endpoints_df = pd.DataFrame(
    HEALTH_ENDPOINTS
)

display(health_endpoints_df)

,name,path,expected_behavior
0,FastAPI liveness,/api/v1/health/live,HTTP 200
1,FastAPI readiness,/api/v1/health/ready,HTTP 200 when ready; structured non-ready resp...
2,Forecast,/api/v1/forecast,HTTP 200 with 72 rows when current artifacts a...
3,Forecast summary,/api/v1/forecast/summary,HTTP 200
4,Alerts,/api/v1/alerts,HTTP 200
5,Metadata,/api/v1/metadata,HTTP 200
6,Streamlit process health,/_stcore/health,HTTP 200


In [6]:
from app.operations.repository_inspection import (
    build_repository_operations_report,
    save_report,
)


repository_operations_report = (
    build_repository_operations_report()
)

repository_operations_report[
    "status"
]

'REPOSITORY_OPERATIONS_INSPECTION_INCOMPLETE'

### Manual Phase 10A review

The automated inspection confirms file and command availability, but the
following items require manual review:

1. whether live inference has a reusable non-notebook command
2. whether AQI and alert generation has a reusable non-notebook command
3. whether Phase 5 and Phase 6 publish artifacts atomically
4. whether all batch reports include a pipeline run ID
5. whether every failed command returns a non-zero exit code
6. whether routine tests avoid real external services
7. whether Docker images include unnecessary notebooks or datasets
8. whether production artifact paths can be redirected through configuration
9. whether any current GitHub workflow duplicates a future cloud scheduler
10. which cloud account, credits, and region are available

In [7]:
live_inference_runner = (
    PROJECT_ROOT
    / "app"
    / "pipelines"
    / "live_inference.py"
)

aqi_pipeline_runner = (
    PROJECT_ROOT
    / "app"
    / "pipelines"
    / "aqi_alert_pipeline.py"
)

identified_gaps = []

if not live_inference_runner.exists():
    identified_gaps.append(
        {
            "code": (
                "LIVE_INFERENCE_COMMAND_MISSING"
            ),
            "description": (
                "The reusable live inference runner "
                "does not exist."
            ),
        }
    )

if not aqi_pipeline_runner.exists():
    identified_gaps.append(
        {
            "code": (
                "AQI_PIPELINE_COMMAND_MISSING"
            ),
            "description": (
                "The reusable AQI and alert runner "
                "does not exist."
            ),
        }
    )

repository_operations_report[
    "identified_gaps"
] = identified_gaps

print(
    "Live inference runner:",
    live_inference_runner,
)

print(
    "AQI pipeline runner:",
    aqi_pipeline_runner,
)

print(
    "Identified gaps:",
    identified_gaps,
)

Live inference runner: /home/riyan/Riyan/projects/pearls-aqi-predictor/app/pipelines/live_inference.py
AQI pipeline runner: /home/riyan/Riyan/projects/pearls-aqi-predictor/app/pipelines/aqi_alert_pipeline.py
Identified gaps: []


In [8]:
REPORT_PATH = save_report(
    repository_operations_report
)

print(
    "Phase 10A report saved:",
    REPORT_PATH,
)

Phase 10A report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_10/repository_operations_report.json
